# 06.07 - PyTorch Tensors and a Tiny Classifier

**Daily output:** a tiny PyTorch classifier trained and evaluated correctly on toy data.

Today covers tensors, device, autograd, `nn.Module`, loss, optimizer, `model.train()`, `model.eval()`, and `torch.no_grad()`.

**Notebook type:** Practice notebook with theory, exercises, and TODO cells.


## Training Loop Contract

A PyTorch training step follows the same rhythm for toy data, CNNs, and transformers:

1. Move input and labels to the model device.
2. Run the forward pass.
3. Compute loss.
4. Clear old gradients with `optimizer.zero_grad()`.
5. Backpropagate with `loss.backward()`.
6. Update weights with `optimizer.step()`.

For `CrossEntropyLoss`, logits are `[batch, classes]` float values, and labels are `[batch]` `torch.long` class IDs.


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


## Tensor Basics

Inspect `.shape`, `.dtype`, `.device`, and `.requires_grad`. Most PyTorch errors are shape, dtype, or device errors.


In [ ]:
# TODO 06-A: Tensor basics.
# Create:
# - x: shape [2, 3], dtype float32
# - y: shape [2], dtype long
# Print shape, dtype, and device.
# Move both tensors to `device`.

raise NotImplementedError("Create and inspect tensors.")


## Autograd

If a tensor has `requires_grad=True`, PyTorch records operations on it. Calling `.backward()` computes gradients for leaf tensors.


In [ ]:
# TODO 06-B: Autograd.
# Create w and b with requires_grad=True.
# Let pred = w * 3 + b and loss = (pred - 10) ** 2.
# Call backward and print gradients.

raise NotImplementedError("Practice autograd.")


## Toy 3-Class Data

This 2D dataset lets us train a classifier quickly while using the same code structure as a real image model.


In [ ]:
# TODO 06-C: Toy 3-class data.
# Implement make_blobs(n_per_class, noise, seed).
# Return shuffled X with shape [N, 2] and y with shape [N].
# Then create train_ds, val_ds, train_loader, and val_loader.

def make_blobs(n_per_class=120, noise=0.65, seed=42):
    raise NotImplementedError

# TODO: create X, y, TensorDataset, and DataLoader objects.


## Model, Loss, Optimizer

`nn.Module` holds trainable layers. A linear classifier learns one score function per class. `CrossEntropyLoss` expects raw logits, not softmax probabilities.


In [ ]:
# TODO 06-D: Model, loss, optimizer.
# Implement TinyLinearClassifier with one nn.Linear(2, 3).
# Create model, CrossEntropyLoss, and SGD optimizer.
# Run one batch through the model and print logits shape/loss.

class TinyLinearClassifier(nn.Module):
    def __init__(self, in_features=2, num_classes=3):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

# TODO: instantiate model, criterion, optimizer, and inspect one forward pass.


## Train and Evaluate Correctly

`model.train()` enables training behavior. `model.eval()` switches to inference behavior. Use `torch.no_grad()` during validation so PyTorch does not store a gradient graph.


In [ ]:
# TODO 06-E: Train/evaluate functions.
# Implement:
# - train_one_epoch(model, loader, criterion, optimizer, device)
# - evaluate(model, loader, criterion, device)
#
# Remember train/eval mode, zero_grad, backward, step, no_grad.

def train_one_epoch(model, loader, criterion, optimizer, device):
    raise NotImplementedError

def evaluate(model, loader, criterion, device):
    raise NotImplementedError


In [ ]:
# TODO 06-F: Training loop.
# Train for 30 epochs and print train/validation loss and accuracy.

raise NotImplementedError("Run the training loop.")


## Macro-F1

Macro-F1 averages F1 across classes, so each class matters equally. This is more informative than accuracy when classes are imbalanced.


In [ ]:
# TODO 06-G: Macro-F1.
# Implement per_class_f1(preds, labels, num_classes).
# Return class, precision, recall, f1, support for each class.
# Then compute macro-F1 on validation predictions.

def per_class_f1(preds, labels, num_classes):
    raise NotImplementedError

# TODO: evaluate model and print per-class F1 plus macro-F1.


## Common PyTorch Bugs

- Labels for `CrossEntropyLoss` must be `torch.long`.
- Logits for multi-class classification must be `[batch, classes]`.
- Model, inputs, and labels must be on the same device.
- Gradients accumulate unless you call `optimizer.zero_grad()`.
- Validation should use `model.eval()` and `torch.no_grad()`.


In [ ]:
# Debug examples: uncomment one at a time and read the error.
# criterion(model(xb.to(device)), yb.float().to(device))
# model.to(device)(xb)  # fails if device is cuda and xb stays on CPU


## Day 06 Checklist

Check input shape, label dtype, logits shape, device consistency, `model.train()` during training, `model.eval()` plus `torch.no_grad()` during validation, gradient clearing, and validation metrics computed on validation data.


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 06 tests passed`.


In [ ]:
def run_day06_tests():
    required_names = [
        "make_blobs",
        "TinyLinearClassifier",
        "train_one_epoch",
        "evaluate",
        "per_class_f1",
    ]
    for name in required_names:
        assert name in globals(), f"Missing function or class: {name}"
        assert callable(globals()[name]), f"{name} must be callable"

    X_test, y_test = make_blobs(n_per_class=8, noise=0.5, seed=123)
    assert X_test.shape == (24, 2), f"Expected X shape (24, 2), got {X_test.shape}"
    assert y_test.shape == (24,), f"Expected y shape (24,), got {y_test.shape}"
    assert X_test.dtype == torch.float32
    assert y_test.dtype == torch.long
    assert set(y_test.tolist()) == {0, 1, 2}

    test_ds = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_ds, batch_size=12, shuffle=False)
    test_model = TinyLinearClassifier().to(device)
    test_criterion = nn.CrossEntropyLoss()
    test_optimizer = torch.optim.SGD(test_model.parameters(), lr=0.1)

    xb, yb = next(iter(test_loader))
    logits = test_model(xb.to(device))
    assert logits.shape == (12, 3), f"Expected logits shape (12, 3), got {logits.shape}"

    train_loss, train_acc = train_one_epoch(test_model, test_loader, test_criterion, test_optimizer, device)
    assert isinstance(train_loss, float)
    assert 0.0 <= train_acc <= 1.0

    metrics = evaluate(test_model, test_loader, test_criterion, device)
    for key in ["loss", "accuracy", "preds", "labels"]:
        assert key in metrics, f"evaluate output missing key: {key}"
    assert len(metrics["preds"]) == len(y_test)
    assert len(metrics["labels"]) == len(y_test)
    assert 0.0 <= metrics["accuracy"] <= 1.0

    perfect_rows = per_class_f1(
        preds=torch.tensor([0, 1, 2, 0, 1, 2]),
        labels=torch.tensor([0, 1, 2, 0, 1, 2]),
        num_classes=3,
    )
    assert len(perfect_rows) == 3
    assert all(abs(row["f1"] - 1.0) < 1e-8 for row in perfect_rows)

    print("Day 06 tests passed")

run_day06_tests()
